In [0]:
-- ============================================================
-- DATA QUALITY VALIDATION CHECKS
-- ============================================================

-- Check 1: Uppercase emails → Expected: 0
SELECT COUNT(*) AS uppercase_email_count
FROM `retail-dwh-project`.silver.DimCustomer
WHERE Email != LOWER(Email);

-- Check 2: UnitPrice = 0 → Expected: 0
SELECT COUNT(*) AS zero_price_count
FROM `retail-dwh-project`.silver.DimProduct
WHERE UnitPrice = 0;

-- Check 3: NULL Region → Expected: 0
SELECT COUNT(*) AS null_region_count
FROM `retail-dwh-project`.silver.DimStore
WHERE Region IS NULL;

-- Check 4: Quantity = 0 in FactSales → Expected: 0
SELECT COUNT(*) AS zero_qty_count
FROM `retail-dwh-project`.silver.FactSales
WHERE Quantity = 0;

-- Check 5: Duplicate TransactionIDs → Expected: 0
SELECT COUNT(*) AS duplicate_txn_count
FROM (
    SELECT TransactionID, COUNT(*) AS cnt
    FROM `retail-dwh-project`.silver.FactSales
    GROUP BY TransactionID
    HAVING cnt > 1
);

-- Check 6: NULL SKs in FactSales → Expected: 0
SELECT COUNT(*) AS null_sk_count
FROM `retail-dwh-project`.silver.FactSales
WHERE CustomerSK IS NULL
   OR ProductSK  IS NULL
   OR StoreSK    IS NULL;

-- Check 7: Duplicate active CustomerIDs → Expected: 0
SELECT CustomerID, COUNT(*) AS cnt
FROM `retail-dwh-project`.silver.DimCustomer
WHERE IsActive = 1
GROUP BY CustomerID
HAVING cnt > 1;

-- Check 8: SCD2 IsActive summary
SELECT IsActive, COUNT(*) AS count
FROM `retail-dwh-project`.silver.DimCustomer
GROUP BY IsActive;

-- Check 9: Amount validation → Expected: PASS
SELECT
    f.TransactionID,
    f.Quantity,
    p.UnitPrice,
    f.Amount,
    ROUND(f.Quantity * p.UnitPrice, 2) AS Expected_Amount,
    CASE
        WHEN f.Amount = ROUND(f.Quantity * p.UnitPrice, 2)
        THEN 'PASS'
        ELSE 'FAIL'
    END AS Amount_Check
FROM `retail-dwh-project`.silver.FactSales f
JOIN `retail-dwh-project`.silver.DimProduct p
    ON f.ProductSK = p.ProductSK
LIMIT 10;